# FFM_2_8 — SYMARE landmark → latent：数据、诊断、缓存

取代 `FFM_2_4_Landmark2Z.ipynb`。**编号重开，旧的 2_4 / 2_5 不要再用。**

## 和 2_4 的区别

| | 2_4（作废） | **2_8** |
|---|---|---|
| latent 来源 | v2.3（三库，不含 SYMARE） | **v2.6（四库，含 SYMARE）** |
| 受试者配对 | `symare61_to_carlab_RESOLVED.csv` 频谱最近邻 | **没有配对**——同一个人 |
| 可用被试 | 52（Craig 已确认配的是错的人） | **61** |
| AE-heldout | 11 人 | **24 人**（val 9 + test 15） |
| 导出 | 只有拍平的 `L_sub` | **结构化 `(N, P, 3)` + 拍平，两个都有** |

Craig 已确认新采集的 CARLab 与 SYMARE 是**完全不同的人**，所以旧的 dCor²ᵤ = 0.071 /
CV-R² = 0.052 / 端到端 4.208 dB **量的不是"几何 → 这个人的 latent"**，一个都不能引用。
它们保留在 `notes/landmark2z/RESULTS_landmark2z.md` 里作为有记录的错误。

## 职责划分

**2_8（本文件）**：受试者名单、landmark 结构化张量、AE latent、诊断、
dCor 去/留判据、**缓存导出**。不拟合任何回归模型。

**2_9**：划分、**fold-local PCA**、三种输入编码（raw / per-landmark RFF /
global RFF）、MLP、超参选择、指标、端到端、对比表。

ridge 也在 2_9，和 MLP 共用同一套 fold-local PCA —— 2_4 的泄漏就是因为
PCA 在 CV 之前用全部被试拟合，两处各有一份逻辑迟早会漂。

## 顺序（便宜的在前，不要跳）

| Cell | 问题 | 要训练吗 |
|---|---|---|
| 6 | latent 还编码"哪只耳"吗 | 否 |
| 7 | **几何和 latent 之间有没有关系** | **否，不拟合任何模型** |
| 8 | 关系在哪个频段 | 否 |
| 9 | 导出缓存给 2_9 | 否 |

In [1]:
# ==========================================================================
# Cell 0 — 环境 · 路径
# ==========================================================================
import os, sys, json, zipfile, hashlib, itertools, warnings
warnings.filterwarnings('ignore')

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
else:
    DRIVE = os.environ.get('ELEC5021_DRIVE', os.path.expanduser('~/Documents/26S2/ELEC5021'))

!pip install healpy dcor -q

BASE = '/content/dataset'
SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'

PATHS = dict(
    landmarks = f'{DRIVE}/ELEC5021/SYMARE/3DShapeData.mat',
    ckpt      = f'{SAVE_DIR}/FFM26_best.pth',          # ★ 四库模型
    split     = f'{SAVE_DIR}/split_v21.json',
    symare_zip= f'{DRIVE}/ELEC5020/SYMARE.zip',
)
for k, p in PATHS.items():
    print(f'{k:11s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先把缺失的文件放到 Drive 上'

# ---- SYMARE 的 .npy（和 FFM_2_6 同一套递归查找）----
def find_dir(root, target):
    c = []
    for dp, _, fns in os.walk(root):
        if '__MACOSX' in dp: continue
        if os.path.basename(dp) == target:
            c.append((sum(f.endswith('.npy') for f in fns), dp))
    c.sort(reverse=True)
    return c[0][1] if c else None

SYM_OUT = f'{BASE}/symare'
os.makedirs(SYM_OUT, exist_ok=True)
if find_dir(SYM_OUT, 'npy_left') is None:
    with zipfile.ZipFile(PATHS['symare_zip']) as z: z.extractall(SYM_OUT)
SYM = dict(left=find_dir(SYM_OUT, 'npy_left'), right=find_dir(SYM_OUT, 'npy_right'),
           suffix=('_left_full.npy', '_right_full.npy'), did=3)
assert SYM['left'] and SYM['right'], '没找到 SYMARE 的 npy_left / npy_right'
print(f"\nSYMARE left  {SYM['left']}  ({len(os.listdir(SYM['left']))} npy)")
print(f"SYMARE right {SYM['right']}  ({len(os.listdir(SYM['right']))} npy)")

MIRROR_SRC = next((p for p in (
    f'{DRIVE}/ELEC5021/analysis/healpix_mirror_idx.npy',
    f'{SAVE_DIR}/healpix_mirror_idx.npy',
    f'{DRIVE}/healpix_mirror_idx.npy') if os.path.exists(p)), None)
print('mirror idx:', MIRROR_SRC or '★缺失，Cell 5 会现场生成')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 148.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 11.7 MB/s eta 0:00:00
landmarks   OK  /content/drive/MyDrive/ELEC5021/SYMARE/3DShapeData.mat
ckpt        OK  /content/drive/MyDrive/HRTF_Checkpoints/FFM26_best.pth
split       OK  /content/drive/MyDrive/HRTF_Checkpoints/split_v21.json
symare_zip  OK  /content/drive/MyDrive/ELEC5020/SYMARE.zip

SYMARE left  /content/dataset/symare/SYMARE/npy_left  (62 npy)
SYMARE right /content/dataset/symare/SYMARE/npy_right  (61 npy)
mirror idx: /content/drive/MyDrive/ELEC5021/analysis/healpix_mirror_idx.npy


In [2]:
# ==========================================================================
# Cell 1 — ★ 配置
# ==========================================================================
CFG = dict(
    # --- landmark 点集 ---
    POINT_SET   = 'ipsi',      # 'ipsi' 同侧耳13 + 中矢面5 + 同侧肩1 = 19点
                               # 'ear'  只要同侧耳 13 点
                               # 'all'  全部 33 点（含对侧耳）
    # --- 目标 z ---
    K_VAR        = 0.95,
    PROJECT_SIDE = 'auto',     # 'auto' 实测 >5% 才投影
    # --- 判据 ---
    N_PERM       = 999,
    SEED         = 0,
)
print(json.dumps(CFG, indent=1))
print('\n★ 坐标的中心化/单位不在这里定 —— 那是编码器的事，放在 2_9。')
print('  本 notebook 导出【原始 mm、左耳坐标系】的结构化张量，2_9 自己变换。')

{
 "POINT_SET": "ipsi",
 "K_VAR": 0.95,
 "PROJECT_SIDE": "auto",
 "N_PERM": 999,
 "SEED": 0
}

★ 坐标的中心化/单位不在这里定 —— 那是编码器的事，放在 2_9。
  本 notebook 导出【原始 mm、左耳坐标系】的结构化张量，2_9 自己变换。


In [3]:
# ==========================================================================
# Cell 2 - 模型定义（与 FFM_2_3 / FFM_2_6 逐字相同，构建时复制进来）
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

model defined (FFM + FiLM)


In [4]:
# ==========================================================================
# Cell 3 — landmark：结构化张量 (61, 33, 3)
#
# 3DShapeData.mat 的分块（由 y 坐标实测确认，SYMARE readme 完全没有记载）：
#   0..12   一侧耳廓 13 点   (y ≈ +79 mm)
#   13..17  中矢面 5 点      (|y| ≈ 2.6 mm)
#   18..30  另一侧耳廓 13 点 (y ≈ -79 mm)，与 0..12 一一对应 (i <-> i+18)
#   31,32   一对左右对称点   (y = -31.8 / +35.1)
# ==========================================================================
import numpy as np, scipy.io as sio

mat = sio.loadmat(PATHS['landmarks'], struct_as_record=False, squeeze_me=True)
LM_ALL = np.stack(list(mat['A'])).astype(np.float64)          # (61, 33, 3) mm
T_MS, T_SS = mat['T_MS'], mat['T_SS']                          # 两个模板，本文件不使用
N_SUBJ_LM, N_LM, C_LM = LM_ALL.shape
assert (N_LM, C_LM) == (33, 3) and not np.isnan(LM_ALL).any()
print(f'LM_ALL {LM_ALL.shape}  (受试者, landmark, xyz)  单位 mm')

EAR_P  = list(range(0, 13))     # y>0 侧耳
MIDSAG = list(range(13, 18))
EAR_N  = list(range(18, 31))    # y<0 侧耳
PAIR_N, PAIR_P = 31, 32

# ✅ 已用声学实测判定（Cell 3b 会重跑）：hL 的能量集中在 y>0，18/18 个受试者
#    的 hL 主方向都指向 0..12 块。SYMARE readme 对此只字未提。
Y_POSITIVE_IS_LEFT = True

_y = LM_ALL[..., 1]
print(f'块 y 均值: ear_p {_y[:, EAR_P].mean():+.1f} | midsag {_y[:, MIDSAG].mean():+.1f} '
      f'| ear_n {_y[:, EAR_N].mean():+.1f} | pair {_y[:, PAIR_N].mean():+.1f}/{_y[:, PAIR_P].mean():+.1f}')
_L, _R = LM_ALL[:, EAR_P], LM_ALL[:, EAR_N].copy(); _R[..., 1] *= -1
print(f'左右耳镜像后逐轴平均差 {np.abs(_L - _R).mean((0, 1)).round(2)} mm  ← 真实的左右不对称')
_cs = np.linalg.norm(LM_ALL - LM_ALL.mean(1, keepdims=True), axis=(1, 2))
print(f'centroid size {_cs.mean():.1f} ± {_cs.std():.1f} mm ({_cs.min():.0f}–{_cs.max():.0f})'
      '  ← 真实的头尺寸差异，★不要归一化掉')
print(f'质心跨被试 std {LM_ALL.mean(1).std(0).round(2)} mm  ← 已在公共坐标系，不需要 Procrustes')

LM_ALL (61, 33, 3)  (受试者, landmark, xyz)  单位 mm
块 y 均值: ear_p +78.9 | midsag +0.5 | ear_n -78.8 | pair -31.8/+35.1
左右耳镜像后逐轴平均差 [1.48 1.57 1.52] mm  ← 真实的左右不对称
centroid size 510.8 ± 21.3 mm (460–567)  ← 真实的头尺寸差异，★不要归一化掉
质心跨被试 std [1.46 0.65 1.41] mm  ← 已在公共坐标系，不需要 Procrustes


In [5]:
# ==========================================================================
# Cell 3b — 左右手别的声学判定（可选，缺 SYMARE/HRIRs 就跳过）
#
# hL / hR 的宽带能量加权主方向 vs landmark 两个耳块的方向。
# 已测 18/18 一致：⟨y⟩_L = +0.495±0.031，⟨y⟩_R = -0.494±0.034，
# y>0 半球 ILD = +12.34±2.03 dB。
# ==========================================================================
SYM_HRIR = f'{DRIVE}/ELEC5021/SYMARE/HRIRs'
if os.path.isdir(f'{SYM_HRIR}/Acoustic'):
    import glob, re
    _a = sio.loadmat(f'{SYM_HRIR}/Parameters/azim.mat')
    _e = sio.loadmat(f'{SYM_HRIR}/Parameters/elev.mat')
    _az = _a[[k for k in _a if not k.startswith('__')][0]].ravel()
    _el = _e[[k for k in _e if not k.startswith('__')][0]].ravel()
    D = np.stack([np.cos(_el)*np.cos(_az), np.cos(_el)*np.sin(_az), np.sin(_el)], 1)
    ok = tot = 0
    for f in sorted(glob.glob(f'{SYM_HRIR}/Acoustic/HA*.mat')):
        k = int(re.search(r'HA(\d+)', f).group(1)); P = LM_ALL[k-1]
        cP, cN = P[EAR_P].mean(0), P[EAR_N].mean(0); mid = (cP+cN)/2
        eP, eN = (cP-mid)/np.linalg.norm(cP-mid), (cN-mid)/np.linalg.norm(cN-mid)
        m = sio.loadmat(f); wL = (m['hL']**2).sum(0); wR = (m['hR']**2).sum(0)
        dL = (D*(wL/wL.sum())[:, None]).sum(0); dL /= np.linalg.norm(dL)
        dR = (D*(wR/wR.sum())[:, None]).sum(0); dR /= np.linalg.norm(dR)
        ok += (dL @ eP > dL @ eN) and (dR @ eN > dR @ eP); tot += 1
    print(f'声学判定 {ok}/{tot} 支持「块 0..12 = 左耳」')
    assert ok == tot and Y_POSITIVE_IS_LEFT, '★ 判定不一致，停下来查'
else:
    print(f'[跳过] 没有 {SYM_HRIR}/Acoustic —— 沿用已验证结论（18/18）')

[跳过] 没有 /content/drive/MyDrive/ELEC5021/SYMARE/HRIRs/Acoustic —— 沿用已验证结论（18/18）


In [6]:
# ==========================================================================
# Cell 4 — 受试者：直接从 split_v21.json 的 SYMARE 键取
#
# ★ 没有映射表。SYM_HA01..61 就是 landmark 的编号，也是 latent 的编号。
#   v2.4 靠 symare61_to_carlab_RESOLVED.csv 配对，Craig 已确认那配的是错的人。
# ==========================================================================
SPLIT = json.load(open(PATHS['split']))
assert 'SYMARE' in SPLIT, ('split_v21.json 里没有 SYMARE 键 —— '
                           '先跑 FFM_2_6 的 Cell 3 生成划分')
part_of = {s: p for p in ('train', 'val', 'test') for s in SPLIT['SYMARE'][p]}

sfxL, sfxR = SYM['suffix']
ids_disk = sorted({f[:-len(sfxL)] for f in os.listdir(SYM['left']) if f.endswith(sfxL)})

SUBJ = []
for sid in ids_disk:
    m = __import__('re').fullmatch(r'SYM_HA(\d{2})', sid)
    assert m, f'编号格式不认识: {sid}（预期 SYM_HA01..SYM_HA61）'
    ha = int(m.group(1))
    assert 1 <= ha <= N_SUBJ_LM, f'{sid} 超出 landmark 的 {N_SUBJ_LM} 个受试者'
    assert sid in part_of, f'{sid} 不在 split_v21.json 的 SYMARE 划分里'
    SUBJ.append(dict(sym=sid, ha=ha, ae_part=part_of[sid]))
SUBJ.sort(key=lambda s: s['ha'])

# ---- §7 要求的映射表。现在三列同源，assert 退化成哨兵，但留着 ----
from collections import Counter
assert len({s['sym'] for s in SUBJ}) == len(SUBJ), '受试者编号有重复'
assert len({s['ha'] for s in SUBJ}) == len(SUBJ),  'landmark 索引有重复'
_p = {k: set(SPLIT['SYMARE'][k]) for k in ('train', 'val', 'test')}
assert not (_p['train'] & _p['val']) and not ((_p['train'] | _p['val']) & _p['test']), '划分有重叠'

print(f"{'HRTF id':<12}{'landmark id':<14}{'latent id':<12}{'库':<9}{'划分':<6}")
def _row(s):
    lm = '…' if s['ha'] == '…' else f"A[{s['ha']-1}]"
    print(f"{s['sym']:<12}{lm:<14}{s['sym']:<12}{'SYMARE':<9}{s['ae_part']:<6}")
for s in SUBJ[:3]: _row(s)
_row(dict(sym='…', ha='…', ae_part='…'))
for s in SUBJ[-2:]: _row(s)
print(f"\n{len(SUBJ)} 人 | AE 分区 {dict(Counter(s['ae_part'] for s in SUBJ))}")
HELDOUT = [s for s in SUBJ if s['ae_part'] != 'train']
print(f"AE-heldout = {len(HELDOUT)} 人（2_4 时代只有 11 人）")
SUBJ_HASH = hashlib.md5(','.join(s['sym'] for s in SUBJ).encode()).hexdigest()[:12]
print(f'受试者名单 hash {SUBJ_HASH}')

HRTF id     landmark id   latent id   库        划分    
SYM_HA01    A[0]          SYM_HA01    SYMARE   train 
SYM_HA02    A[1]          SYM_HA02    SYMARE   train 
SYM_HA03    A[2]          SYM_HA03    SYMARE   test  
…           …             …           SYMARE   …     
SYM_HA60    A[59]         SYM_HA60    SYMARE   train 
SYM_HA61    A[60]         SYM_HA61    SYMARE   train 

61 人 | AE 分区 {'train': 37, 'test': 15, 'val': 9}
AE-heldout = 24 人（2_4 时代只有 11 人）
受试者名单 hash 5d151b755ad8


In [7]:
# ==========================================================================
# Cell 5 — 载入 v2.6，编码 SYMARE 的 latent
#
# ★ encoder.forward 只收 x。数据集条件化是【编码前的逆 FiLM】，做在
#   FFM_OSLO_Autoencoder.forward 里。直接 model.encoder(x) 会把测量染色
#   编进 z —— 而 z 正是下游要用的东西。
# ==========================================================================
import torch, healpy as hp

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ckpt = torch.load(PATHS['ckpt'], map_location=device, weights_only=False)
n_ds, n_bins = ckpt['model_state_dict']['decoder.film.beta'].shape
print(f"checkpoint: epoch {ckpt['epoch']+1} | val {ckpt['val_rmse']:.4f} dB | {n_ds} 库 / {n_bins} bin")
assert n_bins == 93, f'num_bins={n_bins}，不是 93'
assert n_ds == 4, f'{n_ds} 库 —— 需要四库模型（FFM26），三库的没有 SYMARE'
assert ckpt['split']['SYMARE'] == SPLIT['SYMARE'], 'checkpoint 里的 SYMARE 划分和磁盘上的不一致'

model = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                             num_datasets=n_ds, num_bins=n_bins,
                             tie_encoder_film=True).to(device)
model.load_state_dict(ckpt['model_state_dict']); model.eval()
DF = 44100 / 256; freqs = np.arange(n_bins) * DF
LATENT_DIM = 64
del ckpt

if MIRROR_SRC:
    MIRROR = np.load(MIRROR_SRC)
else:
    _th, _ph = hp.pix2ang(16, np.arange(hp.nside2npix(16)))
    MIRROR = hp.ang2pix(16, _th, (-_ph) % (2 * np.pi))
assert len(np.unique(MIRROR)) == len(MIRROR) and np.array_equal(MIRROR[MIRROR], np.arange(len(MIRROR)))

def encode(x_t, did_t):
    xn = model.decoder.film.inverse(x_t, did_t) if model.tie_encoder_film else x_t
    return model.encoder(xn)

def load_ear(sid, side):
    sfx = SYM['suffix'][0 if side == 'left' else 1]
    x = np.load(os.path.join(SYM[side], f'{sid}{sfx}')).astype(np.float32)[:n_bins]
    return x[:, MIRROR] if side == 'right' else x

EARS, Z_list = [], []
with torch.no_grad():
    did = torch.tensor([SYM['did']], device=device)
    for j, s in enumerate(SUBJ):
        for side in ('left', 'right'):
            xt = torch.from_numpy(load_ear(s['sym'], side))[None].to(device)
            z = encode(xt, did)
            if j == 0 and side == 'left':
                assert torch.equal(z, model(xt, did)[1]), 'encode() 与 model.forward 不一致'
                print('encode() 与 model.forward 的 z 逐位一致 ✓')
            EARS.append(dict(**s, side=side)); Z_list.append(z[0].float().cpu().numpy())
Z_fd  = np.stack(Z_list)                       # (2N, 93, 64)
Z_ear = Z_fd.reshape(len(Z_fd), -1)            # (2N, 5952)
side_is_right = np.array([e['side'] == 'right' for e in EARS])
print(f'Z_fd {Z_fd.shape} -> Z_ear {Z_ear.shape} | {len(SUBJ)} 人 x 2 耳')

checkpoint: epoch 99 | val 1.4855 dB | 4 库 / 93 bin
encode() 与 model.forward 的 z 逐位一致 ✓
Z_fd (122, 93, 64) -> Z_ear (122, 5952) | 61 人 x 2 耳


In [8]:
# ==========================================================================
# Cell 6 — latent 诊断（先测，不要照抄任何历史数字）
# ==========================================================================
from scipy import stats

v = Z_ear[side_is_right].mean(0) - Z_ear[~side_is_right].mean(0)
v = v / np.linalg.norm(v)
Z_nos = Z_ear - np.outer(Z_ear @ v, v)
side_frac = 1 - Z_nos.var() / Z_ear.var()
print(f'① side 方向占 latent 方差 {side_frac:.1%}   (v1 71.7% / v2.3 实测 1.4%)')
if side_frac > 0.5:
    print('   ★ 超过 50% —— 镜像可能没生效，停下来查 MonauralHRTFDataset')

do_proj = side_frac > 0.05 if CFG['PROJECT_SIDE'] == 'auto' else bool(CFG['PROJECT_SIDE'])
Z_use = Z_nos if do_proj else Z_ear
print(f'   -> {"投影掉 side 方向" if do_proj else "不投影"}')

# ② AE-train vs AE-heldout 的 latent 有没有系统性差异（★人层面）
def to_subject(M):
    assert len(M) == 2 * len(SUBJ), '行序必须是 每人左、右 交替'
    return M.reshape(len(SUBJ), 2, *M.shape[1:]).mean(1)

tr_m = np.array([s['ae_part'] == 'train' for s in SUBJ])
nrm = np.linalg.norm(to_subject(Z_use), axis=1)
t = stats.ttest_ind(nrm[tr_m], nrm[~tr_m])
print(f'② ||z||  AE-train n={tr_m.sum():2d}: {nrm[tr_m].mean():8.2f} ± {nrm[tr_m].std():6.2f} | '
      f'heldout n={(~tr_m).sum():2d}: {nrm[~tr_m].mean():8.2f} ± {nrm[~tr_m].std():6.2f}')
print(f'   t={t.statistic:+.2f}, p={t.pvalue:.3f} -> '
      f'{"没有可测差异" if t.pvalue > .05 else "★有差异，报数时以 heldout 为准"}')
print('   注：||z|| 只是一维汇总量，是很弱的代理。2_9 的 heldout R² 才是直接检验。')

# ③ 有效维度（★只是这批样本的症状，不是 latent 的内在维度）
Zc = Z_use - Z_use.mean(0)
S = np.linalg.svd(Zc, compute_uv=False)
K_eff = int(np.searchsorted(np.cumsum(S**2) / (S**2).sum(), CFG['K_VAR']) + 1)
print(f'③ 保留 {CFG["K_VAR"]:.0%} 方差需 K={K_eff}（上限 min(2N-1,5952)={min(len(Z_use)-1,5952)}）')
print('   ★ 这是 N 太小的症状，不要当成 latent 的内在维度引用。')

# ④ SYMARE 的被试间方差 —— 基线 4.805 是四库里最低的，人群可能更窄
print(f'④ latent 被试间方差 {to_subject(Z_use).var():.4f}  '
      f'(SYMARE 的 mean-HRTF 基线 4.805 dB 是四库最低 —— 人群更同质，'
      '几何能预测的目标方差本身就小，报 R² 时要说明)')

① side 方向占 latent 方差 0.2%   (v1 71.7% / v2.3 实测 1.4%)
   -> 不投影
② ||z||  AE-train n=37:  4089.69 ± 262.64 | heldout n=24:  4092.06 ± 312.56
   t=-0.03, p=0.975 -> 没有可测差异
   注：||z|| 只是一维汇总量，是很弱的代理。2_9 的 heldout R² 才是直接检验。
③ 保留 95% 方差需 K=87（上限 min(2N-1,5952)=121）
   ★ 这是 N 太小的症状，不要当成 latent 的内在维度引用。
④ latent 被试间方差 2824.7104  (SYMARE 的 mean-HRTF 基线 4.805 dB 是四库最低 —— 人群更同质，几何能预测的目标方差本身就小，报 R² 时要说明)


In [9]:
# ==========================================================================
# Cell 7 — ★★ 去/留判据：dCor²ᵤ + 置换检验（不拟合任何模型，无需划分）
#
# ⚠️ 必须用无偏版。实测：N=57、独立数据上，有偏的 dcor.distance_correlation
#    读 0.830，线性 CKA 读 0.378。零分布只取决于 N。
# ==========================================================================
import dcor
rng = np.random.default_rng(CFG['SEED'])

IPSI = {'ipsi': EAR_P + MIDSAG + [PAIR_P], 'ear': EAR_P,
        'all': list(range(N_LM))}[CFG['POINT_SET']]
P_LM = len(IPSI)
print(f'点集 {CFG["POINT_SET"]!r} -> P = {P_LM} 点\n')

def mirror_config(Pm):
    """镜像到另一侧：y 取反 + 交换两个耳块 + 交换那对对称点"""
    Q = Pm.copy(); Q[:, 1] *= -1
    Q[EAR_P], Q[EAR_N] = Pm[EAR_N] * [1, -1, 1], Pm[EAR_P] * [1, -1, 1]
    Q[PAIR_P], Q[PAIR_N] = Pm[PAIR_N] * [1, -1, 1], Pm[PAIR_P] * [1, -1, 1]
    return Q

xyz_ear = []
for s in SUBJ:
    P0 = LM_ALL[s['ha'] - 1]
    ipsi_left = P0 if Y_POSITIVE_IS_LEFT else mirror_config(P0)
    for cfg_ in (ipsi_left, mirror_config(ipsi_left)):       # left, right
        xyz_ear.append(cfg_[IPSI])
XYZ_EAR = np.stack(xyz_ear)                                   # (2N, P, 3) mm
assert XYZ_EAR.shape == (2 * len(SUBJ), P_LM, 3)
XYZ_SUB = to_subject(XYZ_EAR)                                 # (N, P, 3)
_d = np.abs(XYZ_EAR[0::2] - XYZ_EAR[1::2]).mean()
print(f'XYZ_EAR {XYZ_EAR.shape} | XYZ_SUB {XYZ_SUB.shape} | 单位 mm、左耳坐标系')
print(f'左右耳配置平均 |差| {_d:.2f} mm  ← 真实不对称；为 0 说明镜像写错了')
assert 0.2 < _d < 15, '镜像疑似出错'

def whiten(X, keep=0.95):
    X = X - X.mean(0)
    U, S_, _ = np.linalg.svd(X, full_matrices=False)
    k = int(np.searchsorted(np.cumsum(S_**2) / (S_**2).sum(), keep) + 1)
    return U[:, :k] * np.sqrt(len(X)), k

def dcor_test(L, Z_, nperm, tag):
    obs = dcor.u_distance_correlation_sqr(L, Z_)
    null = np.array([dcor.u_distance_correlation_sqr(L, Z_[rng.permutation(len(Z_))])
                     for _ in range(nperm)])
    p = (1 + (null >= obs).sum()) / (1 + nperm)
    print(f'  {tag:22s} dCor²ᵤ = {obs:+.4f} | 零分布95分位 {np.quantile(null,.95):+.4f} '
          f'| p = {p:.4f} {"✅" if p < .05 else "—"}')
    return obs, p

Ls, kL = whiten(XYZ_SUB.reshape(len(SUBJ), -1))
Zs, kZ = whiten(to_subject(Z_use))
print(f'\n白化后 landmark {kL} 维 | latent {kZ} 维 | N = {len(Ls)} 人（独立样本）')
print('★ 主结果 —— 每人一行:')
OBS, PVAL = dcor_test(Ls, Zs, CFG['N_PERM'], 'subject-level')

print('\n负对照（都应当不显著）:')
dcor_test(Ls, Zs[rng.permutation(len(Zs))], CFG['N_PERM'], 'shuffled z')
dcor_test(rng.normal(size=Ls.shape), Zs, CFG['N_PERM'], 'random features')

print(f'\n判据: {"有信号，继续 Cell 8" if PVAL < .05 else "★ 没有可测信号 —— 这本身就是结果"}')
print('注意 dCor² 会饱和：真实共享方差 10% 就已读到 0.42。它回答「有没有」，不回答「多少」。')

点集 'ipsi' -> P = 19 点

XYZ_EAR (122, 19, 3) | XYZ_SUB (61, 19, 3) | 单位 mm、左耳坐标系
左右耳配置平均 |差| 1.60 mm  ← 真实不对称；为 0 说明镜像写错了

白化后 landmark 15 维 | latent 47 维 | N = 61 人（独立样本）
★ 主结果 —— 每人一行:
  subject-level          dCor²ᵤ = +0.0710 | 零分布95分位 +0.0390 | p = 0.0020 ✅

负对照（都应当不显著）:
  shuffled z             dCor²ᵤ = -0.0083 | 零分布95分位 +0.0382 | p = 0.6090 —
  random features        dCor²ᵤ = -0.0309 | 零分布95分位 +0.0407 | p = 0.9100 —

判据: 有信号，继续 Cell 8
注意 dCor² 会饱和：真实共享方差 10% 就已读到 0.42。它回答「有没有」，不回答「多少」。


In [10]:
# ==========================================================================
# Cell 8 — 信号在哪个频段（科学上最有意思的输出，不是那个单一分数）
# ==========================================================================
EDGES = ((0, 500), (500, 6000), (6000, 12000), (12000, int(freqs[-1]) + 1))
Zf_sub = to_subject(Z_fd.reshape(len(Z_fd), -1)).reshape(-1, n_bins, LATENT_DIM)

print(f'{"band":16s}{"bins":>6}{"dCor²ᵤ":>10}{"p":>8}')
BAND_RES = {}
for lo, hi in EDGES:
    m = (freqs >= lo) & (freqs < hi)
    Zb, _ = whiten(Zf_sub[:, m, :].reshape(len(Zf_sub), -1))
    o = dcor.u_distance_correlation_sqr(Ls, Zb)
    null = np.array([dcor.u_distance_correlation_sqr(Ls, Zb[rng.permutation(len(Zb))])
                     for _ in range(199)])
    p = (1 + (null >= o).sum()) / 200
    BAND_RES[f'{lo}-{hi}'] = (float(o), float(p))
    print(f'{f"{lo/1000:.1f}–{hi/1000:.1f} kHz":16s}{int(m.sum()):>6}{o:>10.4f}{p:>8.3f}'
          f' {"✅" if p < .05 else ""}')
print('\n对照：重建残差/基线 的分带比值 DC<500Hz 59.3%(最差) / 6–12kHz 25.3%(最好)。')
print('若信号只在 6 kHz 以上 —— 那是耳廓控制的频段，不是头尺寸（头尺寸会出现在低频）。')
print('⚠️ DC–0.5 kHz 只有 3 个 bin 且有已知病灶，那一格是在说数据质量，不是几何。')

band              bins    dCor²ᵤ       p
0.0–0.5 kHz          3    0.1399   0.005 ✅
0.5–6.0 kHz         32    0.1282   0.005 ✅
6.0–12.0 kHz        35    0.0631   0.015 ✅
12.0–15.8 kHz       23    0.0725   0.005 ✅

对照：重建残差/基线 的分带比值 DC<500Hz 59.3%(最差) / 6–12kHz 25.3%(最好)。
若信号只在 6 kHz 以上 —— 那是耳廓控制的频段，不是头尺寸（头尺寸会出现在低频）。
⚠️ DC–0.5 kHz 只有 3 个 bin 且有已知病灶，那一格是在说数据质量，不是几何。


In [11]:
# ==========================================================================
# Cell 9 — ★ 导出缓存给 FFM_2_9
#
# 结构化张量和拍平的都存。2_9 的 per-landmark RFF 需要 (N, P, 3)，
# 绝对不要从拍平的向量去猜 (P,3)。
# ==========================================================================
CACHE = f'{SAVE_DIR}/landmark2z_v28_cache.npz'
np.savez_compressed(
    CACHE,
    # --- 几何：原始 mm、左耳坐标系，中心化/单位换算留给 2_9 ---
    xyz_ear=XYZ_EAR.astype(np.float32),        # (2N, P, 3)
    xyz_sub=XYZ_SUB.astype(np.float32),        # (N,  P, 3)
    L_ear=XYZ_EAR.reshape(len(XYZ_EAR), -1).astype(np.float32),   # (2N, 3P)
    L_sub=XYZ_SUB.reshape(len(XYZ_SUB), -1).astype(np.float32),   # (N,  3P)
    # --- latent ---
    Z_ear=Z_ear.astype(np.float32), Z_sub=to_subject(Z_use).astype(np.float32),
    Z_fd_sub=Zf_sub.astype(np.float32),        # (N, 93, 64) 分带用
    # --- 元数据 ---
    sym=np.array([s['sym'] for s in SUBJ]),
    ha=np.array([s['ha'] for s in SUBJ]),
    ae_part=np.array([s['ae_part'] for s in SUBJ]),
    side_is_right=side_is_right,
    point_set=CFG['POINT_SET'], point_idx=np.array(IPSI),
    P=P_LM, n_bins=n_bins, latent_dim=LATENT_DIM, freqs=freqs,
    coord_unit='mm', coord_frame='left-ear, SYMARE common frame, uncentred',
    side_frac=side_frac, side_projected=do_proj,
    dcor_obs=OBS, dcor_p=PVAL, band_res=json.dumps(BAND_RES),
    subject_hash=SUBJ_HASH, ckpt=os.path.basename(PATHS['ckpt']),
    split=json.dumps(SPLIT['SYMARE']),
)
print(f'缓存 -> {CACHE}')
print(f'  xyz_sub {XYZ_SUB.shape} | Z_sub {to_subject(Z_use).shape} | {len(SUBJ)} 人')
print(f'  point_set={CFG["POINT_SET"]!r} P={P_LM} | 单位 mm | hash {SUBJ_HASH}')
print(f'  AE 分区 {dict(Counter(s["ae_part"] for s in SUBJ))}')
print(f'  dCor²ᵤ={OBS:+.4f} p={PVAL:.4f} | side {side_frac:.1%} 投影={do_proj}')
print('\n★ 2_9 从这个文件开始，不用重跑编码器。')

缓存 -> /content/drive/MyDrive/HRTF_Checkpoints/landmark2z_v28_cache.npz
  xyz_sub (61, 19, 3) | Z_sub (61, 5952) | 61 人
  point_set='ipsi' P=19 | 单位 mm | hash 5d151b755ad8
  AE 分区 {'train': 37, 'test': 15, 'val': 9}
  dCor²ᵤ=+0.0710 p=0.0020 | side 0.2% 投影=False

★ 2_9 从这个文件开始，不用重跑编码器。


## 交付前检查

1. **Cell 5 的 `n_ds == 4` 断言**——必须是四库模型。三库的 `FFM23_best.pth` 里没有
   SYMARE，跑了也是错的。
2. **Cell 6 ① 的 side 方差**应当在 1.4% 附近。跳回 70% 说明镜像坏了。
3. **Cell 6 ④**：SYMARE 的 mean-HRTF 基线 4.805 dB 是四库里**最低**的，说明这 61 人
   彼此更像。目标方差本身就小，R² 偏低时要能区分「几何没信息」和「这批人差异小」。
4. **Cell 7 的负对照**必须都不显著。shuffled z 若显著就说明管线泄漏。
5. **不要引用 2_4 时代的任何数字**。那批是用错误配对算的。

## 与旧结果的关系

`notes/landmark2z/RESULTS_landmark2z.md` 里的 dCor²ᵤ=0.071 / CV-R²=0.052 /
端到端 4.208 dB **全部作废**，但保留为有记录的错误。本 notebook 的结果是第一次
真正回答「几何 → **这个人的** latent」。